In [1]:
# Cell 1: Kiểm tra thư viện
import sys
import pandas as pd
import numpy as np

print(f"Python : {sys.version}")
print(f"Pandas : {pd.__version__}")
print(f"Numpy  : {np.__version__}")

Python : 3.12.5 (tags/v3.12.5:ff3bc82, Aug  6 2024, 20:45:27) [MSC v.1940 64 bit (AMD64)]
Pandas : 3.0.0
Numpy  : 2.4.0


In [2]:
# Cell 2: Đường dẫn tất cả file trong dự án
from pathlib import Path

# Thư mục gốc
ROOT = Path(r"D:\hocki7\Data Annalyst\Detai")

# Input — data gốc (không bao giờ ghi đè)
PATH_TRANSACTION = ROOT / "1_data" / "raw" / "train_transaction.csv"
PATH_IDENTITY    = ROOT / "1_data" / "raw" / "train_identity.csv"

# Output — từng bước lưu ra file riêng
PATH_MERGED      = ROOT / "1_data" / "processed" / "01_merged_raw.csv"
PATH_SAMPLED     = ROOT / "1_data" / "processed" / "02_sampled.csv"

# Kiểm tra file gốc có tồn tại không
print("=== KIỂM TRA FILE GỐC ===")
for p in [PATH_TRANSACTION, PATH_IDENTITY]:
    status = "✅ Tìm thấy" if p.exists() else "❌ KHÔNG tìm thấy"
    size   = f"{p.stat().st_size / 1024**2:.0f} MB" if p.exists() else ""
    print(f"  {status} | {size:>8} | {p.name}")

=== KIỂM TRA FILE GỐC ===
  ✅ Tìm thấy |   652 MB | train_transaction.csv
  ✅ Tìm thấy |    25 MB | train_identity.csv


In [3]:
# Cell 3: Đọc train_transaction.csv — chỉ load cột cần thiết
# usecols giúp tiết kiệm RAM vì file gốc có 394 cột

COLS_TRANSACTION = [
    # --- Khóa & Nhãn ---
    'TransactionID',        # khóa để merge với identity
    'isFraud',              # nhãn: 1=gian lận, 0=bình thường

    # --- Thời gian ---
    'TransactionDT',        # giây kể từ điểm gốc (cần convert sang giờ/ngày)

    # --- Số tiền ---
    'TransactionAmt',       # số tiền giao dịch — có outlier rất mạnh

    # --- Sản phẩm & Thẻ ---
    'ProductCD',            # loại sản phẩm: W/H/C/S/R — categorical
    'card1',                # mã số thẻ — numeric nhưng là ID
    'card4',                # thương hiệu thẻ: visa/mastercard/... — categorical
    'card6',                # loại thẻ: debit/credit — categorical

    # --- Địa chỉ ---
    'addr1',                # mã vùng người gửi — có missing
    'addr2',                # mã quốc gia — có missing

    # --- Khoảng cách địa lý ---
    'dist1',                # khoảng cách 1 — ~47% missing
    'dist2',                # khoảng cách 2 — ~65% missing

    # --- Email ---
    'P_emaildomain',        # email người gửi: gmail.com/yahoo.com/...
    'R_emaildomain',        # email người nhận

    # --- Đặc trưng hành vi (counting) ---
    'C1',                   # số lần thẻ xuất hiện
    'C2',                   # số địa chỉ khác nhau của thẻ
    'C6',                   # số thiết bị khác nhau

    # --- Đặc trưng thời gian (timedelta) ---
    'D1',                   # số ngày kể từ giao dịch trước
    'D4',                   # số ngày kể từ lần đầu giao dịch
    'D10',                  # số ngày kể từ giao dịch đầu tiên của thiết bị

    # --- Đặc trưng so khớp ---
    'M4',                   # kết quả khớp thông tin: M0/M1/M2 — categorical
    'M6',                   # kết quả khớp khác: T/F — binary categorical
]

print("Đang đọc train_transaction.csv ...")
print(f"Chỉ load {len(COLS_TRANSACTION)} / 394 cột")
print("-" * 40)

df_trans = pd.read_csv(PATH_TRANSACTION, usecols=COLS_TRANSACTION)

print(f"✅ Đọc xong!")
print(f"   Số dòng : {df_trans.shape[0]:,}")
print(f"   Số cột  : {df_trans.shape[1]}")
print(f"   RAM dùng: {df_trans.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nKiểu dữ liệu từng cột:")
print(df_trans.dtypes)

Đang đọc train_transaction.csv ...
Chỉ load 22 / 394 cột
----------------------------------------
✅ Đọc xong!
   Số dòng : 590,540
   Số cột  : 22
   RAM dùng: 258.3 MB

Kiểu dữ liệu từng cột:
TransactionID       int64
isFraud             int64
TransactionDT       int64
TransactionAmt    float64
ProductCD             str
card1               int64
card4                 str
card6                 str
addr1             float64
addr2             float64
dist1             float64
dist2             float64
P_emaildomain         str
R_emaildomain         str
C1                float64
C2                float64
C6                float64
D1                float64
D4                float64
D10               float64
M4                    str
M6                    str
dtype: object


In [4]:
# Cell 4: Đọc train_identity.csv

COLS_IDENTITY = [
    'TransactionID',    # khóa merge
    'DeviceType',       # desktop/mobile — categorical có null
    'DeviceInfo',       # tên thiết bị thô — rất bẩn, cần làm sạch
    'id_12',            # found/not found — categorical
    'id_15',            # new/found/unknown — categorical
    'id_28',            # new/found — categorical
    'id_31',            # tên trình duyệt — text bẩn (chrome/firefox/...)
]

print("Đang đọc train_identity.csv ...")
df_iden = pd.read_csv(PATH_IDENTITY, usecols=COLS_IDENTITY)

print(f"✅ Đọc xong!")
print(f"   Số dòng : {df_iden.shape[0]:,}")
print(f"   Số cột  : {df_iden.shape[1]}")
print(f"   RAM dùng: {df_iden.memory_usage(deep=True).sum()/1024**2:.1f} MB")
print(f"\nKiểu dữ liệu:")
print(df_iden.dtypes)

Đang đọc train_identity.csv ...
✅ Đọc xong!
   Số dòng : 144,233
   Số cột  : 7
   RAM dùng: 47.3 MB

Kiểu dữ liệu:
TransactionID    int64
id_12              str
id_15              str
id_28              str
id_31              str
DeviceType         str
DeviceInfo         str
dtype: object


In [5]:
# Cell 5: Merge theo TransactionID bằng LEFT JOIN
# LEFT JOIN: giữ toàn bộ transaction
# Nếu transaction không có trong identity → các cột identity = NaN
# (thực tế: chỉ ~50% giao dịch có thông tin thiết bị)

print("Đang merge 2 file ...")
df_merged = pd.merge(df_trans, df_iden, on='TransactionID', how='left')

print(f"✅ Merge xong!")
print(f"   Trước merge : transaction={df_trans.shape[0]:,} | identity={df_iden.shape[0]:,}")
print(f"   Sau merge   : {df_merged.shape[0]:,} dòng | {df_merged.shape[1]} cột")

# Kiểm tra tỷ lệ có identity
co_identity = df_merged['DeviceType'].notna().sum()
print(f"\n   Giao dịch CÓ thông tin thiết bị : {co_identity:,} ({co_identity/len(df_merged)*100:.1f}%)")
print(f"   Giao dịch KHÔNG có thiết bị     : {len(df_merged)-co_identity:,} ({(len(df_merged)-co_identity)/len(df_merged)*100:.1f}%)")

Đang merge 2 file ...
✅ Merge xong!
   Trước merge : transaction=590,540 | identity=144,233
   Sau merge   : 590,540 dòng | 28 cột

   Giao dịch CÓ thông tin thiết bị : 140,810 (23.8%)
   Giao dịch KHÔNG có thiết bị     : 449,730 (76.2%)


In [6]:
# Cell 6: Lấy mẫu 200,000 dòng — giữ đúng tỷ lệ fraud

df_fraud    = df_merged[df_merged['isFraud'] == 1]
df_notfraud = df_merged[df_merged['isFraud'] == 0]

print("=== PHÂN PHỐI NHÃN TRONG DATA GỐC ===")
print(f"  Fraud (1)     : {len(df_fraud):,} dòng ({len(df_fraud)/len(df_merged)*100:.2f}%)")
print(f"  Không fraud(0): {len(df_notfraud):,} dòng ({len(df_notfraud)/len(df_merged)*100:.2f}%)")

# Cấu hình mẫu
TONG_MAU    = 200_000
TY_LE_FRAUD = 0.20        # giữ 20% fraud để mô hình học đủ pattern

so_fraud    = min(len(df_fraud), int(TONG_MAU * TY_LE_FRAUD))
so_notfraud = TONG_MAU - so_fraud

# Lấy mẫu
sample_fraud    = df_fraud.sample(n=so_fraud, random_state=42)
sample_notfraud = df_notfraud.sample(n=so_notfraud, random_state=42)

df_sample = (pd.concat([sample_fraud, sample_notfraud])
               .sample(frac=1, random_state=42)
               .reset_index(drop=True))

print(f"\n=== MẪU SAU KHI LẤY ===")
print(f"  Fraud    : {so_fraud:,} dòng (20%)")
print(f"  Không fraud : {so_notfraud:,} dòng (80%)")
print(f"  Tổng     : {len(df_sample):,} dòng")

=== PHÂN PHỐI NHÃN TRONG DATA GỐC ===
  Fraud (1)     : 20,663 dòng (3.50%)
  Không fraud(0): 569,877 dòng (96.50%)

=== MẪU SAU KHI LẤY ===
  Fraud    : 20,663 dòng (20%)
  Không fraud : 179,337 dòng (80%)
  Tổng     : 200,000 dòng


In [9]:
# Cell 7: Tổng quan missing values và lưu file

print("=" * 60)
print("     BÁO CÁO TỔNG QUAN DATASET SAU KHI LẤY MẪU")
print("=" * 60)

missing = df_sample.isnull().sum()
pct     = (missing / len(df_sample) * 100).round(1)

# Tự động phân loại từng cột — không hardcode số lượng
def phan_loai_cot(col, dtype, missing_pct):
    if col == 'isFraud':
        return '🎯 NHÃN MỤC TIÊU'
    elif col == 'TransactionID':
        return '🔑 Khóa chính'
    elif col == 'TransactionDT':
        return '⏱ Cần convert → giờ/ngày'
    elif col == 'TransactionAmt':
        return '💰 Có outlier → cần log transform'
    elif missing_pct > 50:
        return f'⚠️ Missing nhiều ({missing_pct}%) — cân nhắc xử lý'
    elif missing_pct > 20:
        return f'⚠️ Missing vừa ({missing_pct}%) — cần impute'
    elif missing_pct > 0:
        return f'ℹ️ Missing ít ({missing_pct}%) — cần impute'
    elif dtype == 'object':
        return '📝 Categorical — cần encode'
    else:
        return '🔢 Numeric — OK'

ghi_chu = [
    phan_loai_cot(col, str(df_sample[col].dtype), pct[col])
    for col in df_sample.columns
]

report = pd.DataFrame({
    'Cột'                : df_sample.columns,
    'Kiểu dữ liệu'       : df_sample.dtypes.values,
    'Số thiếu'           : missing.values,
    'Tỷ lệ thiếu (%)'    : pct.values,
    'Ghi chú'            : ghi_chu
}).sort_values('Tỷ lệ thiếu (%)', ascending=False).reset_index(drop=True)

# In ra đẹp hơn
pd.set_option('display.max_colwidth', 45)
pd.set_option('display.max_rows', 30)
print(report.to_string(index=False))

# Thống kê nhanh
print(f"\n{'='*60}")
print(f"TỔNG KẾT:")
print(f"  Tổng số cột        : {df_sample.shape[1]}")
print(f"  Tổng số dòng       : {df_sample.shape[0]:,}")
print(f"  Cột có missing     : {(pct > 0).sum()} / {df_sample.shape[1]} cột")
print(f"  Cột missing > 50%  : {(pct > 50).sum()} cột — cần xem xét bỏ")
print(f"  Cột categorical    : {(df_sample.dtypes == 'object').sum()} cột — cần encode")
print(f"  Cột numeric        : {(df_sample.dtypes != 'object').sum()} cột")
print(f"  Phân phối nhãn     :")
print(f"    Fraud (1)        : {(df_sample['isFraud']==1).sum():,} ({(df_sample['isFraud']==1).mean()*100:.1f}%)")
print(f"    Không fraud (0)  : {(df_sample['isFraud']==0).sum():,} ({(df_sample['isFraud']==0).mean()*100:.1f}%)")

# Lưu file
print(f"\n{'='*60}")
print("Đang lưu file 02_sampled.csv ...")
df_sample.to_csv(PATH_SAMPLED, index=False)
print(f"✅ Đã lưu: {PATH_SAMPLED}")
print(f"   Kích thước: {PATH_SAMPLED.stat().st_size/1024**2:.1f} MB")
print(f"\n→ Bước tiếp theo: mở 02_EDA.ipynb")

     BÁO CÁO TỔNG QUAN DATASET SAU KHI LẤY MẪU
           Cột Kiểu dữ liệu  Số thiếu  Tỷ lệ thiếu (%)                                   Ghi chú
         dist2      float64    185528             92.8 ⚠️ Missing nhiều (92.8%) — cân nhắc xử lý
    DeviceInfo          str    156498             78.2 ⚠️ Missing nhiều (78.2%) — cân nhắc xử lý
 R_emaildomain          str    148805             74.4 ⚠️ Missing nhiều (74.4%) — cân nhắc xử lý
         id_31          str    147951             74.0 ⚠️ Missing nhiều (74.0%) — cân nhắc xử lý
    DeviceType          str    147760             73.9 ⚠️ Missing nhiều (73.9%) — cân nhắc xử lý
         id_15          str    147685             73.8 ⚠️ Missing nhiều (73.8%) — cân nhắc xử lý
         id_28          str    147690             73.8 ⚠️ Missing nhiều (73.8%) — cân nhắc xử lý
         id_12          str    146617             73.3 ⚠️ Missing nhiều (73.3%) — cân nhắc xử lý
         dist1      float64    122014             61.0 ⚠️ Missing nhiều (61.0%) 